## Full Near-real-time (NRT) pipeline

### Data Loading
* load local Data
* download new timestamp (recent date)
* merge dataset

### Run NRT Detection

In [1]:
# imports
from pathlib import Path

import xarray as xr
import geopandas as gpd

from water_timeseries.breakpoint import NRTBreakpoint
from water_timeseries.dataset import DWDataset
from water_timeseries.downloader import EarthEngineDownloader

## Load data

In [2]:
def dw_test_dataset():
    """Load the real Dynamic World test dataset.

    Returns the actual DW dataset from tests/data/lakes_dw_test.nc
    """
    test_data_dir = Path("../tests/data")
    dw_file = test_data_dir / "lakes_dw_test.nc"
    return xr.open_dataset(dw_file)

Load pre loaded historic Dynamic World Dataset

In [3]:
dw_ds_historical = DWDataset(dw_test_dataset())

Show valid ids and dates
* the last available date is June 2025 (2025-06-01)
* There are 5 lakes in the dataset

In [4]:
print(dw_ds_historical.object_ids_)
print(dw_ds_historical.dates_[-1])

['b7uefy0bvcrc', 'b7veemx7k1v0', 'b7g4rf3n3x43', 'b7g6dd4vhjjy', 'b7g1rrchqs18']
2025-06-01T00:00:00.000000000



## Data Download
Now we need to append our timeseries

#### Load and match vector polygons for download
* load vector dataset
* confirm overlapping id_geohash values between ds and vector file

In [5]:
vector_path = "../tests/data/lakes_polygons_test.parquet"
gdf = gpd.read_parquet(vector_path)

In [6]:
gdf_filtered = gdf[gdf["id_geohash"].isin(dw_ds_historical.object_ids_)]
gdf_filtered["id_geohash"].tolist()

['b7uefy0bvcrc',
 'b7g6dd4vhjjy',
 'b7g4rf3n3x43',
 'b7g1rrchqs18',
 'b7veemx7k1v0']

#### Download data
* first setup the downloader using a dedicated GEE project name
* **Please insert your GEE project name**
* download dynamic world data for the five lakes for July 2025
* the downloaded dataset is a xr.dataset

In [7]:
ee_project = "YOUR_EE_PROJECT_NAME"
downloader = EarthEngineDownloader(ee_project=ee_project)

Initializing EarthEngineDownloader with project: pdg-wg-workflow
EarthEngineDownloader initialized successfully. Output directory: downloads


In [8]:
ds_downloaded = downloader.download_dw_monthly(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    date_list=["2025-07"],
    id_list=gdf_filtered["id_geohash"].tolist(),
)
ds_downloaded

Initial dataset has 5 features
Applying ID filter: 5 IDs specified
After ID filter: 5 features
No spatial bbox filtering applied (using default global bounds)
Processing 5 features
Processing 5 features x 1 dates = 5 total requests
Processing dates: ['2025-07-01']
Chunking: n_features=5, max_total_requests=500, n_dates=1, features_per_chunk=5
Split 5 features into 1 chunks (chunk_size=5)
Processing 1 chunks with 1 workers


Download complete: 5 items, 1 dates collected


<xarray.Dataset> Size: 408B
Dimensions:             (id_geohash: 5, date: 1)
Coordinates:
  * id_geohash          (id_geohash) object 40B 'b7g1rrchqs18' ... 'b7veemx7k...
  * date                (date) datetime64[ns] 8B 2025-07-01
Data variables:
    bare                (id_geohash, date) float64 40B 0.0 0.0 0.0 0.2 0.0
    built               (id_geohash, date) float64 40B 0.0 0.0 0.0 0.43 0.0
    crops               (id_geohash, date) float64 40B 0.0 0.0 0.0 0.01 0.0
    flooded_vegetation  (id_geohash, date) float64 40B 0.0 0.36 0.0 0.71 0.0
    grass               (id_geohash, date) float64 40B 12.41 7.929 ... 1.261
    shrub_and_scrub     (id_geohash, date) float64 40B 3.151 36.42 ... 0.2371
    snow_and_ice        (id_geohash, date) int64 40B 0 0 0 0 0
    trees               (id_geohash, date) float64 40B 6.689 326.9 ... 2.505
    water               (id_geohash, date) float64 40B 7.106 5.958 ... 32.47
Attributes:
    description:                  This datasets provides the monthly area of ...
    accompanying vector dataset:  lakes_polygons_test.parquet
    source:                       https://github.com/PermafrostDiscoveryGatew...
    author:                       Ingmar Nitze (Alfred Wegener Institute), Ka...
    contact:                      ingmar.nitze@awi.de

In [9]:
dw_ds_new = DWDataset(ds_downloaded)

#### Merge datasets

* We will convert it to DWDataset
* both DW dataset (historical + new) can be simply merged

In [10]:
dw_ds_merged = dw_ds_historical.merge(dw_ds_new)

The datasets were concatenated where the new date was appended,
* checking the latest 2 dates shows that the new date was appended
* object ids (id_geohash / individual lakes) stay the same

In [11]:
print(dw_ds_merged.dates_[-2:])
print(dw_ds_merged.object_ids_)

[numpy.datetime64('2025-06-01T00:00:00.000000000'), numpy.datetime64('2025-07-01T00:00:00.000000000')]
['b7g1rrchqs18', 'b7g4rf3n3x43', 'b7g6dd4vhjjy', 'b7uefy0bvcrc', 'b7veemx7k1v0']


Lets visualize if appending the dataset worked

In [12]:
dw_ds_merged.plot_timeseries_interactive(id_geohash="b7uefy0bvcrc")

### Near Real-time analysis
* NRT analysis uses ARIMA to determine if a specified date (typically the latest date) is an outlier compared to the previous time-series

In [13]:
bp_nrt = NRTBreakpoint()

Here we can check the ARIMA prediction for all 5 lakes
* water_residual values, which indicate the difference from the predicted water area are all close to zero
* water_predicted is in all cases within the confidence range ( > water_predicted_lower_90, < water_predicted_upper_90 )
* there is no meaningful lake drainage event

In [14]:
bp_nrtbreaks_simple_dw_2018 = bp_nrt.calculate_break(
    dataset=dw_ds_merged, analysis_date="2025-07", data_aggregation_period="all"
)
bp_nrtbreaks_simple_dw_2018

2025-07-01 00:00:00
2025-07


NRT breakpoints: 100%|██████████| 5/5 [00:00<00:00, 280.28it/s]
/isipd/projects/Response/GIS_RS_projects/Ingmar_other/water-timeseries-v2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/isipd/projects/Response/GIS_RS_projects/Ingmar_other/water-timeseries-v2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/isipd/projects/Response/GIS_RS_projects/Ingmar_other/water-timeseries-v2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/isipd/projects/Response/GI

,date,water_observed,water_predicted,water_predicted_lower_90,water_predicted_upper_90,water_residual,water_historical_mean,water_historical_median,water_historical_std,water_historical_min,water_historical_max
id_geohash,,,,,,,,,,,
b7g1rrchqs18,2025-07-01,0.2420,0.2394,0.1896,0.2892,0.0026,0.5477,0.5612,0.1950,0.2648,0.8226
b7g4rf3n3x43,2025-07-01,0.0158,0.0232,-0.1516,0.1980,-0.0074,0.2622,0.0412,0.3915,0.0179,0.9948
b7g6dd4vhjjy,2025-07-01,0.0000,-0.0549,-0.3157,0.2059,0.0549,0.3095,0.0146,0.3853,0.0000,0.8830
b7uefy0bvcrc,2025-07-01,0.0115,0.0153,-0.2289,0.2595,-0.0038,0.2399,0.0396,0.3904,0.0046,0.9952
b7veemx7k1v0,2025-07-01,0.8902,0.9223,0.7020,1.1426,-0.0321,0.7514,0.9088,0.2329,0.3419,0.9996


#### Geospatial postprocessing
* We can join the output on the input vector layer

In [15]:
# full join keeping all lakes in the gdf, even those without breakpoints (will have NaN values for breakpoint columns)
joined_all_2018 = gdf_filtered.set_index("id_geohash").join(bp_nrtbreaks_simple_dw_2018)
joined_all_2018

,label_id,id,Area_start_ha,Area_end_ha,NetChange_ha,NetChange_perc,GrossIncrease_ha,GrossIncrease_perc,GrossDecrease_ha,GrossDecrease_perc,...,water_observed,water_predicted,water_predicted_lower_90,water_predicted_upper_90,water_residual,water_historical_mean,water_historical_median,water_historical_std,water_historical_min,water_historical_max
id_geohash,,,,,,,,,,,,,,,,,,,,,
b7uefy0bvcrc,18405,18405,177.036305,44.042402,-132.993903,-75.122390,0.3285,0.185555,133.322403,302.713742,...,0.0115,0.0153,-0.2289,0.2595,-0.0038,0.2399,0.0396,0.3904,0.0046,0.9952
b7g6dd4vhjjy,18915,18915,3.306600,1.901700,-1.404900,-42.487753,0.1296,3.919434,1.534500,80.690965,...,0.0000,-0.0549,-0.3157,0.2059,0.0549,0.3095,0.0146,0.3853,0.0000,0.8830
b7g4rf3n3x43,19110,19110,364.293031,148.437903,-215.855128,-59.253159,4.6377,1.273069,220.492828,148.542134,...,0.0158,0.0232,-0.1516,0.1980,-0.0074,0.2622,0.0412,0.3915,0.0179,0.9948
b7g1rrchqs18,20282,20282,29.235599,15.457500,-13.778100,-47.127817,0.1305,0.446374,13.908600,89.979623,...,0.2420,0.2394,0.1896,0.2892,0.0026,0.5477,0.5612,0.1950,0.2648,0.8226
b7veemx7k1v0,69811,69811,36.305100,20.202300,-16.102799,-44.354098,0.0927,0.255336,16.195499,80.166611,...,0.8902,0.9223,0.7020,1.1426,-0.0321,0.7514,0.9088,0.2329,0.3419,0.9996


#### Filter
* filter to data with significant lake area loss in the defined month
* results in an empty dataframe in our case

In [16]:
# filter to lakes with a significant negative breakpoint (water loss) in the last month
joined_all_2018.query("water_residual < -0.25")

,label_id,id,Area_start_ha,Area_end_ha,NetChange_ha,NetChange_perc,GrossIncrease_ha,GrossIncrease_perc,GrossDecrease_ha,GrossDecrease_perc,...,water_observed,water_predicted,water_predicted_lower_90,water_predicted_upper_90,water_residual,water_historical_mean,water_historical_median,water_historical_std,water_historical_min,water_historical_max
id_geohash,,,,,,,,,,,,,,,,,,,,,


### Analysis for a know break data
* we know that one lake drained in June 2018 - so let's try this date

In [17]:
bp_nrtbreaks_simple_dw_2018 = bp_nrt.calculate_break(
    dataset=dw_ds_merged, analysis_date="2018-06", data_aggregation_period="all"
)
# full join keeping all lakes in the gdf, even those without breakpoints (will have NaN values for breakpoint columns)
joined_all_2018 = gdf_filtered.set_index("id_geohash").join(bp_nrtbreaks_simple_dw_2018)

# filter to lakes with a significant negative breakpoint (water loss) in the last month
joined_all_2018.query("water_residual < -0.25")

2018-06-01 00:00:00
2018-06


NRT breakpoints: 100%|██████████| 4/4 [00:00<00:00,  5.72it/s]
/isipd/projects/Response/GIS_RS_projects/Ingmar_other/water-timeseries-v2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/isipd/projects/Response/GIS_RS_projects/Ingmar_other/water-timeseries-v2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/isipd/projects/Response/GIS_RS_projects/Ingmar_other/water-timeseries-v2/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
[Parallel(n_jobs=4)]: Done  

,label_id,id,Area_start_ha,Area_end_ha,NetChange_ha,NetChange_perc,GrossIncrease_ha,GrossIncrease_perc,GrossDecrease_ha,GrossDecrease_perc,...,water_observed,water_predicted,water_predicted_lower_90,water_predicted_upper_90,water_residual,water_historical_mean,water_historical_median,water_historical_std,water_historical_min,water_historical_max
id_geohash,,,,,,,,,,,,,,,,,,,,,
b7uefy0bvcrc,18405,18405,177.036305,44.042402,-132.993903,-75.12239,0.3285,0.185555,133.322403,302.713742,...,0.1831,0.9905,0.9873,0.9936,-0.8074,0.9914,0.9917,0.0037,0.9848,0.9952


We can explore the result and save it to a parquet file or other vector format

In [18]:
# filter relevant columns for output
columns = ["id_geohash"] + bp_nrtbreaks_simple_dw_2018.columns.tolist() + ["geometry"]
# make query and reset index to get id_geohash back as a column, then select relevant columns for output
output_df = joined_all_2018.query("water_residual < -0.25").reset_index()[columns]

In [19]:
# visualize output
output_df.explore()

In [20]:
output_df.to_parquet("nrt_breakpoints_lakes_2018-06.parquet")